# YZM5109 – Assignment I · Option C: LoRA Fine-Tuning vs. Prompt Engineering

**Model:** `Qwen/Qwen2.5-1.5B-Instruct` (decoder-only) · **Task / data:** grade-school math word problems, `openai/gsm8k` (exact-match on the final number)

| Condition | Model | Prompt |
|---|---|---|
| **C1** | original | zero-shot direct |
| **C2** | original | zero-shot chain-of-thought |
| **C3** | original | Program-Aided LM (model writes Python, we execute it) |
| **C4** | LoRA fine-tuned (r=16, α=32) | *exactly* the C1 direct prompt |
| C5 (optional) | original | 4-shot with worked solutions |

All conditions share the same test items and the same greedy decoding (`do_sample=False`, `max_new_tokens=512`).

**How to run:** `Runtime → Change runtime type → T4 GPU` (or better), then `Runtime → Run all`.
Run once with `QUICK_RUN = True` (≈5 min) to check everything works, then set it to `False` for the real run.
Predictions are written to Google Drive after every batch, so if Colab disconnects just run all cells again; finished work is skipped.

In [ ]:
# Tested versions (same versions the pipeline was smoke-tested with).
!pip -q install "transformers==5.22.1" "peft==0.21.2" "datasets==5.1.0" accelerate

In [ ]:
%%writefile a1lib.py
"""YZM5109 Assignment I, Option C: LoRA fine-tuning vs. prompt engineering on GSM8K.

Self-contained helper library. The Colab notebook embeds this exact file, and
tests/smoke_test.py exercises it end to end with a tiny randomly initialised model.
"""
from __future__ import annotations

import ast
import json
import math
import os
import random
import re
import subprocess
import sys
import tempfile
import time
from dataclasses import asdict, dataclass, field
from typing import Callable

import torch

# --------------------------------------------------------------------------------------
# Prompts. Every condition uses the same system prompt and the same answer-format line,
# so the only thing that changes between C1, C2 and C3 is the prompting technique.
# --------------------------------------------------------------------------------------
SYSTEM_PROMPT = "You are a helpful assistant."
ANSWER_FORMAT = 'End your response with "The answer is: <number>".'

DIRECT_TEMPLATE = "Solve the following math problem.\n\nQuestion: {question}\n\n" + ANSWER_FORMAT

COT_TEMPLATE = (
    "Solve the following math problem. Let's think step by step: break the problem into "
    "smaller steps, write out every intermediate calculation, and check the result before "
    "answering.\n\nQuestion: {question}\n\n" + ANSWER_FORMAT
)

PAL_TEMPLATE = (
    "Write a Python function named `solution()` that solves the following math problem and "
    "returns the final numeric answer. Use descriptive variable names and simple arithmetic. "
    "Do not use input() or external libraries. Reply with a single ```python code block only."
    "\n\nQuestion: {question}"
)


def _msgs(user: str) -> list[dict]:
    return [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}]


def build_direct(item: dict) -> list[dict]:
    return _msgs(DIRECT_TEMPLATE.format(question=item["question"]))


def build_cot(item: dict) -> list[dict]:
    return _msgs(COT_TEMPLATE.format(question=item["question"]))


def build_pal(item: dict) -> list[dict]:
    return _msgs(PAL_TEMPLATE.format(question=item["question"]))


def make_fewshot_builder(exemplars: list[dict]) -> Callable[[dict], list[dict]]:
    """Optional extra condition: k-shot with worked solutions as prior chat turns."""

    def build(item: dict) -> list[dict]:
        msgs = [{"role": "system", "content": SYSTEM_PROMPT}]
        for ex in exemplars:
            msgs.append({"role": "user", "content": DIRECT_TEMPLATE.format(question=ex["question"])})
            msgs.append({"role": "assistant", "content": format_target(ex)})
        msgs.append({"role": "user", "content": DIRECT_TEMPLATE.format(question=item["question"])})
        return msgs

    return build


# --------------------------------------------------------------------------------------
# GSM8K parsing and answer extraction.
# --------------------------------------------------------------------------------------
_CALC_ANNOT = re.compile(r"<<[^>]*>>")
_NUM = r"-?\d[\d,]*(?:\.\d+)?"


def parse_gsm8k(row: dict) -> dict:
    rationale, gold = row["answer"].split("####")
    return {
        "question": row["question"].strip(),
        "rationale": _CALC_ANNOT.sub("", rationale).strip(),
        "gold": normalize_number(gold),
    }


def format_target(item: dict) -> str:
    """Fine-tuning target: GSM8K rationale (calculator tags stripped) plus the shared answer line."""
    return f"{item['rationale']}\nThe answer is: {item['gold']}"


def normalize_number(s: str | None) -> str | None:
    if s is None:
        return None
    s = s.strip().replace(",", "").replace("$", "").rstrip(".")
    if not s:
        return None
    try:
        v = float(s)
    except ValueError:
        return None
    if math.isnan(v) or math.isinf(v):
        return None
    if abs(v - round(v)) < 1e-9:
        return str(int(round(v)))
    return f"{v:.6g}"


def extract_answer(text: str) -> str | None:
    """Prefer the mandated 'The answer is: X' line, then \\boxed{X}, then the last number."""
    m = re.findall(r"answer is[:\s]*\**\s*\$?\s*(" + _NUM + ")", text, flags=re.IGNORECASE)
    if m:
        return normalize_number(m[-1])
    m = re.findall(r"\\boxed\{\s*\$?\s*(" + _NUM + ")", text)
    if m:
        return normalize_number(m[-1])
    m = re.findall(_NUM, text)
    return normalize_number(m[-1]) if m else None


def is_correct(pred: str | None, gold: str | None) -> bool:
    if pred is None or gold is None:
        return False
    try:
        return abs(float(pred) - float(gold)) < 1e-4
    except ValueError:
        return False


# --------------------------------------------------------------------------------------
# PAL: execute model-written code in an isolated subprocess with a timeout.
# --------------------------------------------------------------------------------------
_CODE_BLOCK = re.compile(r"```(?:python|py)?\s*\n(.*?)```", flags=re.DOTALL | re.IGNORECASE)


def extract_code(text: str) -> str:
    blocks = _CODE_BLOCK.findall(text)
    if blocks:
        return max(blocks, key=len)
    # Unterminated block (hit max_new_tokens) or bare code.
    return re.sub(r"^```(?:python|py)?\s*\n", "", text.strip())


def _print_last_expression(code: str) -> str:
    """Mimic a notebook/REPL: if the program ends with a bare expression (e.g. `total_cost`),
    print its value. Models frequently end PAL programs this way instead of calling print()."""
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return code
    if tree.body and isinstance(tree.body[-1], ast.Expr):
        value = tree.body[-1].value
        is_print = (isinstance(value, ast.Call) and isinstance(value.func, ast.Name)
                    and value.func.id == "print")
        if not is_print:
            tree.body[-1] = ast.Expr(ast.Call(func=ast.Name("print", ast.Load()), args=[value], keywords=[]))
            ast.fix_missing_locations(tree)
            return ast.unparse(tree)
    return code


def run_pal(text: str, timeout: float = 5.0) -> tuple[str | None, str]:
    """Answer = return value of solution() if defined, else the last printed value or bare final expression."""
    code = _print_last_expression(extract_code(text))
    runner = (
        code
        + "\n\nif __name__ == '__main__':\n"
        + "    if callable(globals().get('solution')):\n"
        + "        print(solution())\n"
    )
    with tempfile.TemporaryDirectory() as tmp:
        try:
            proc = subprocess.run(
                [sys.executable, "-I", "-c", runner],
                capture_output=True,
                text=True,
                timeout=timeout,
                cwd=tmp,
                stdin=subprocess.DEVNULL,
            )
        except subprocess.TimeoutExpired:
            return None, "timeout"
    if proc.returncode != 0:
        err = proc.stderr.strip().splitlines()
        return None, ("error: " + err[-1]) if err else "error"
    lines = [ln for ln in proc.stdout.strip().splitlines() if ln.strip()]
    if not lines:
        return None, "no_output"
    m = re.findall(_NUM, lines[-1])
    return (normalize_number(m[-1]) if m else None), "ok"


def score_text(text: str) -> tuple[str | None, str]:
    return extract_answer(text), "ok"


# --------------------------------------------------------------------------------------
# Batched greedy generation with OOM back-off.
# --------------------------------------------------------------------------------------
def make_generation_config(model, tok, max_new_tokens: int):
    from transformers import GenerationConfig

    eos = tok.eos_token_id
    eos_ids = sorted({i for i in [eos, tok.convert_tokens_to_ids("<|im_end|>"), tok.convert_tokens_to_ids("<|endoftext|>")]
                      if isinstance(i, int) and i >= 0 and i != tok.unk_token_id})
    return GenerationConfig(
        do_sample=False,
        temperature=None,
        top_p=None,
        top_k=None,
        max_new_tokens=max_new_tokens,
        pad_token_id=tok.pad_token_id,
        eos_token_id=eos_ids,
        repetition_penalty=1.0,
    )


@torch.no_grad()
def generate_batch(model, tok, msg_lists: list[list[dict]], gen_cfg) -> list[tuple[str, int]]:
    texts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in msg_lists]
    enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
    out = model.generate(**enc, generation_config=gen_cfg)
    new = out[:, enc["input_ids"].shape[1]:]
    eos_ids = set(gen_cfg.eos_token_id if isinstance(gen_cfg.eos_token_id, list) else [gen_cfg.eos_token_id])
    results = []
    for row in new.tolist():
        n = 0
        for t in row:
            n += 1
            if t in eos_ids:
                break
        results.append((tok.decode(row, skip_special_tokens=True).strip(), n))
    return results


def run_condition(
    name: str,
    model,
    tok,
    items: list[dict],
    build_msgs: Callable[[dict], list[dict]],
    scorer: Callable[[str], tuple[str | None, str]],
    out_dir: str,
    batch_size: int = 16,
    max_new_tokens: int = 512,
    log_every: int = 1,
) -> list[dict]:
    """Evaluate one condition. Predictions stream to JSONL so a Colab disconnect can resume."""
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, f"{name}.jsonl")
    done: dict[int, dict] = {}
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                r = json.loads(line)
                done[r["idx"]] = r
    todo = [i for i in range(len(items)) if i not in done]
    if not todo:
        print(f"[{name}] already complete ({len(done)} items), loading from {path}")
        return [done[i] for i in range(len(items))]

    gen_cfg = make_generation_config(model, tok, max_new_tokens)
    model.eval()
    # Sort by prompt length so batches pad less; results are re-ordered by idx at the end.
    todo.sort(key=lambda i: -len(build_msgs(items[i])[-1]["content"]))
    bs, pos, t_start, n_batches = batch_size, 0, time.time(), 0
    with open(path, "a") as f:
        while pos < len(todo):
            chunk = todo[pos:pos + bs]
            try:
                t0 = time.time()
                outs = generate_batch(model, tok, [build_msgs(items[i]) for i in chunk], gen_cfg)
                dt = time.time() - t0
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if bs == 1:
                    raise
                bs = max(1, bs // 2)
                print(f"[{name}] OOM, lowering batch size to {bs}")
                continue
            for i, (text, ntok) in zip(chunk, outs):
                pred, status = scorer(text)
                rec = {
                    "idx": i,
                    "condition": name,
                    "question": items[i]["question"],
                    "gold": items[i]["gold"],
                    "output": text,
                    "pred": pred,
                    "status": status,
                    "correct": is_correct(pred, items[i]["gold"]),
                    "gen_tokens": ntok,
                    "hit_max_tokens": ntok >= max_new_tokens,
                    "latency_s": dt / len(chunk),
                }
                done[i] = rec
                f.write(json.dumps(rec) + "\n")
            f.flush()
            pos += len(chunk)
            n_batches += 1
            if n_batches % log_every == 0:
                acc = sum(done[j]["correct"] for j in done) / len(done)
                print(f"[{name}] {len(done)}/{len(items)}  running acc={acc:.3f}  "
                      f"elapsed={time.time() - t_start:.0f}s  bs={bs}")
    return [done[i] for i in range(len(items))]


# --------------------------------------------------------------------------------------
# LoRA fine-tuning (plain PyTorch loop: fewer moving parts than Trainer across versions).
# --------------------------------------------------------------------------------------
@dataclass
class LoraTrainConfig:
    r: int = 16
    alpha: int = 32
    dropout: float = 0.05
    target_modules: list[str] = field(default_factory=lambda: [
        "q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
    lr: float = 2e-4
    micro_batch_size: int = 4
    grad_accum: int = 4
    max_steps: int = 300
    warmup_ratio: float = 0.05
    max_len: int = 512
    weight_decay: float = 0.0
    max_grad_norm: float = 1.0
    seed: int = 42
    gradient_checkpointing: bool = True
    log_every: int = 10


def tokenize_sft(tok, item: dict, max_len: int) -> dict | None:
    """Tokenise one (direct prompt -> rationale) pair; loss only on the assistant tokens."""
    prompt_msgs = build_direct(item)
    prompt_text = tok.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)
    full_text = tok.apply_chat_template(
        prompt_msgs + [{"role": "assistant", "content": format_target(item)}], tokenize=False)
    if not full_text.startswith(prompt_text):
        raise ValueError("Chat template does not extend the generation prompt; cannot mask labels safely.")
    p_ids = tok(prompt_text, add_special_tokens=False)["input_ids"]
    c_ids = tok(full_text[len(prompt_text):], add_special_tokens=False)["input_ids"]
    ids = (p_ids + c_ids)[:max_len]
    labels = ([-100] * len(p_ids) + c_ids)[:max_len]
    if all(x == -100 for x in labels):
        return None
    return {"input_ids": ids, "labels": labels}


def _collate(batch: list[dict], pad_id: int) -> dict:
    L = max(len(b["input_ids"]) for b in batch)
    ids = torch.full((len(batch), L), pad_id, dtype=torch.long)
    labels = torch.full((len(batch), L), -100, dtype=torch.long)
    mask = torch.zeros((len(batch), L), dtype=torch.long)
    for k, b in enumerate(batch):
        n = len(b["input_ids"])
        ids[k, :n] = torch.tensor(b["input_ids"])
        labels[k, :n] = torch.tensor(b["labels"])
        mask[k, :n] = 1
    return {"input_ids": ids, "labels": labels, "attention_mask": mask}


def count_params(model) -> tuple[int, int]:
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total


def train_lora(model, tok, train_items: list[dict], cfg: LoraTrainConfig, out_dir: str | None = None):
    from peft import LoraConfig, get_peft_model

    random.seed(cfg.seed)
    torch.manual_seed(cfg.seed)
    lcfg = LoraConfig(r=cfg.r, lora_alpha=cfg.alpha, lora_dropout=cfg.dropout,
                      target_modules=cfg.target_modules, bias="none", task_type="CAUSAL_LM")
    if cfg.gradient_checkpointing:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model.enable_input_require_grads()
        model.config.use_cache = False
    model = get_peft_model(model, lcfg)
    trainable, total = count_params(model)
    print(f"Trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.3f}%)")

    data = [x for x in (tokenize_sft(tok, it, cfg.max_len) for it in train_items) if x is not None]
    device = next(model.parameters()).device
    base_dtype = next(p for p in model.parameters() if not p.requires_grad).dtype
    use_amp = device.type == "cuda" and base_dtype in (torch.float16, torch.bfloat16)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp and base_dtype == torch.float16)

    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg.lr, weight_decay=cfg.weight_decay)
    warmup = max(1, int(cfg.warmup_ratio * cfg.max_steps))

    def lr_lambda(step):
        if step < warmup:
            return (step + 1) / warmup
        progress = (step - warmup) / max(1, cfg.max_steps - warmup)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    order: list[int] = []
    rng = random.Random(cfg.seed)

    def next_micro():
        nonlocal order
        if len(order) < cfg.micro_batch_size:
            fresh = list(range(len(data)))
            rng.shuffle(fresh)
            order += fresh
        idx, order = order[:cfg.micro_batch_size], order[cfg.micro_batch_size:]
        return _collate([data[i] for i in idx], tok.pad_token_id)

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    model.train()
    log, t0, seen_tokens = [], time.time(), 0
    for step in range(cfg.max_steps):
        opt.zero_grad(set_to_none=True)
        step_loss = 0.0
        for _ in range(cfg.grad_accum):
            batch = {k: v.to(device) for k, v in next_micro().items()}
            seen_tokens += int(batch["attention_mask"].sum())
            with torch.autocast(device_type=device.type, dtype=base_dtype, enabled=use_amp):
                loss = model(**batch).loss / cfg.grad_accum
            scaler.scale(loss).backward()
            step_loss += loss.item()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, cfg.max_grad_norm)
        scaler.step(opt)
        scaler.update()
        sched.step()
        log.append({"step": step + 1, "loss": step_loss, "lr": sched.get_last_lr()[0]})
        if (step + 1) % cfg.log_every == 0 or step == 0:
            print(f"step {step + 1}/{cfg.max_steps}  loss={step_loss:.4f}  lr={sched.get_last_lr()[0]:.2e}  "
                  f"elapsed={time.time() - t0:.0f}s")
    if device.type == "cuda":
        torch.cuda.synchronize()
    train_time = time.time() - t0

    model.eval()
    if cfg.gradient_checkpointing:
        model.base_model.model.gradient_checkpointing_disable()
    model.config.use_cache = True
    stats = {
        "lora": asdict(cfg),
        "trainable_params": trainable,
        "total_params": total,
        "trainable_pct": 100 * trainable / total,
        "train_time_s": train_time,
        "train_examples_available": len(data),
        "examples_seen": cfg.max_steps * cfg.micro_batch_size * cfg.grad_accum,
        "tokens_seen": seen_tokens,
        "peak_gpu_mem_gb": (torch.cuda.max_memory_allocated() / 1e9) if device.type == "cuda" else None,
        "loss_log": log,
    }
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
        model.save_pretrained(os.path.join(out_dir, "lora_adapter"))
        with open(os.path.join(out_dir, "train_stats.json"), "w") as f:
            json.dump(stats, f, indent=2)
    return model, stats


# --------------------------------------------------------------------------------------
# Statistics: accuracy, bootstrap CI, exact McNemar test on paired predictions.
# --------------------------------------------------------------------------------------
def bootstrap_ci(correct: list[bool], n_boot: int = 10000, seed: int = 0, alpha: float = 0.05):
    rng = random.Random(seed)
    n = len(correct)
    accs = sorted(sum(correct[rng.randrange(n)] for _ in range(n)) / n for _ in range(n_boot))
    return accs[int(alpha / 2 * n_boot)], accs[int((1 - alpha / 2) * n_boot) - 1]


def mcnemar_exact(a: list[bool], b: list[bool]) -> dict:
    """Two-sided exact McNemar test. b01: a wrong & b right, b10: a right & b wrong."""
    b01 = sum((not x) and y for x, y in zip(a, b))
    b10 = sum(x and (not y) for x, y in zip(a, b))
    n = b01 + b10
    if n == 0:
        return {"b01": 0, "b10": 0, "p_value": 1.0}
    k = min(b01, b10)
    p = sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n
    return {"b01": b01, "b10": b10, "p_value": min(1.0, 2 * p)}


def summarize(results: dict[str, list[dict]], n_boot: int = 10000) -> list[dict]:
    rows = []
    for name, recs in results.items():
        corr = [r["correct"] for r in recs]
        lo, hi = bootstrap_ci(corr, n_boot=n_boot)
        n = len(recs)
        rows.append({
            "condition": name,
            "n": n,
            "accuracy": sum(corr) / n,
            "ci_low": lo,
            "ci_high": hi,
            "mean_gen_tokens": sum(r["gen_tokens"] for r in recs) / n,
            "truncated_pct": 100 * sum(r["hit_max_tokens"] for r in recs) / n,
            "no_answer_pct": 100 * sum(r["pred"] is None for r in recs) / n,
            "mean_latency_s": sum(r["latency_s"] for r in recs) / n,
        })
    return rows


def pairwise_tests(results: dict[str, list[dict]]) -> list[dict]:
    names = list(results)
    out = []
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a = [r["correct"] for r in results[names[i]]]
            b = [r["correct"] for r in results[names[j]]]
            t = mcnemar_exact(a, b)
            out.append({"a": names[i], "b": names[j], **t,
                        "delta_acc": (sum(b) - sum(a)) / len(a)})
    return out


def latex_results_table(rows: list[dict], labels: dict[str, str] | None = None) -> str:
    labels = labels or {}
    lines = [
        r"\begin{table}[t]", r"\centering",
        r"\caption{GSM8K test accuracy (greedy decoding, identical settings). 95\% bootstrap CI.}",
        r"\label{tab:main}", r"\begin{tabular}{lccc}", r"\toprule",
        r"Condition & Acc. (\%) & 95\% CI & Avg. tokens \\", r"\midrule",
    ]
    for r in rows:
        label = labels.get(r["condition"], r["condition"].replace("_", r"\_"))
        lines.append(f"{label} & {100 * r['accuracy']:.1f} & "
                     f"[{100 * r['ci_low']:.1f}, {100 * r['ci_high']:.1f}] & {r['mean_gen_tokens']:.0f} \\\\")
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    return "\n".join(lines)

## 1 · Configuration

In [ ]:
QUICK_RUN   = True          # True: 20 test items + 10 training steps, to verify the pipeline. Set False for the real run.
USE_DRIVE   = True          # store results on Google Drive so a disconnect does not lose work
RUN_FEWSHOT = False         # optional 5th condition (4-shot); adds ~1 condition worth of runtime

MODEL_ID    = "Qwen/Qwen2.5-1.5B-Instruct"
N_TEST      = None          # None = full GSM8K test set (1319). Use e.g. 500 if you are short on GPU time.
SEED        = 42
GEN_BATCH   = 32            # lowered automatically on OOM
MAX_NEW_TOK = 512           # identical for every condition

import a1lib as L
LORA_CFG = L.LoraTrainConfig(
    r=16, alpha=32, dropout=0.05,
    lr=2e-4, micro_batch_size=4, grad_accum=4,   # effective batch 16
    max_steps=300, max_len=512, seed=SEED,
)

if QUICK_RUN:
    N_TEST, LORA_CFG.max_steps, LORA_CFG.log_every = 20, 10, 2

RUN_NAME = f"{MODEL_ID.split('/')[-1]}_n{N_TEST or 'all'}_steps{LORA_CFG.max_steps}" + ("_quick" if QUICK_RUN else "")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = f"/content/drive/MyDrive/YZM5109_A1/{RUN_NAME}"
else:
    OUT_DIR = f"/content/YZM5109_A1/{RUN_NAME}"
import os; os.makedirs(OUT_DIR, exist_ok=True)
print("Results ->", OUT_DIR)

## 2 · Data

In [ ]:
import random, json, torch
from datasets import load_dataset

ds = load_dataset("openai/gsm8k", "main")
train_all = [L.parse_gsm8k(r) for r in ds["train"]]
test_all  = [L.parse_gsm8k(r) for r in ds["test"]]

rng = random.Random(SEED)
test_items = test_all if N_TEST is None else rng.sample(test_all, N_TEST)

# Few-shot exemplars come from the training split, never from the test split.
fewshot_exemplars = random.Random(SEED + 1).sample(train_all, 4)
print(f"train={len(train_all)}  test(eval)={len(test_items)}")
print(test_items[0])

## 3 · Original model: C1–C3 (prompting only)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

major, _ = torch.cuda.get_device_capability() if torch.cuda.is_available() else (0, 0)
DTYPE = torch.bfloat16 if major >= 8 else torch.float16     # T4 (sm75) -> fp16, L4/A100 -> bf16
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE, device_map="cuda")
model.eval()

ENV = {"gpu": torch.cuda.get_device_name(0), "dtype": str(DTYPE), "torch": torch.__version__,
       "model": MODEL_ID, "n_test": len(test_items), "max_new_tokens": MAX_NEW_TOK, "seed": SEED}
print(ENV)
print(L.count_params(model))

In [ ]:
results = {}
results["C1_direct"] = L.run_condition("C1_direct", model, tok, test_items, L.build_direct, L.score_text,
                                       OUT_DIR, batch_size=GEN_BATCH, max_new_tokens=MAX_NEW_TOK)

In [ ]:
results["C2_cot"] = L.run_condition("C2_cot", model, tok, test_items, L.build_cot, L.score_text,
                                    OUT_DIR, batch_size=GEN_BATCH, max_new_tokens=MAX_NEW_TOK)

In [ ]:
results["C3_pal"] = L.run_condition("C3_pal", model, tok, test_items, L.build_pal, L.run_pal,
                                    OUT_DIR, batch_size=GEN_BATCH, max_new_tokens=MAX_NEW_TOK)

In [ ]:
if RUN_FEWSHOT:
    results["C5_fewshot"] = L.run_condition("C5_fewshot", model, tok, test_items,
                                            L.make_fewshot_builder(fewshot_exemplars), L.score_text,
                                            OUT_DIR, batch_size=max(1, GEN_BATCH // 2), max_new_tokens=MAX_NEW_TOK)

## 4 · C4: LoRA fine-tuning, evaluated with the C1 direct prompt
Training input = the C1 direct prompt; target = GSM8K reference solution (calculator tags removed) + `The answer is: N`.
Loss is computed only on the answer tokens. If an adapter already exists in `OUT_DIR` (e.g. after a disconnect) it is reloaded instead of retrained.

In [ ]:
from peft import PeftModel
adapter_dir = os.path.join(OUT_DIR, "lora_adapter")
stats_path  = os.path.join(OUT_DIR, "train_stats.json")
if os.path.exists(os.path.join(adapter_dir, "adapter_config.json")) and os.path.exists(stats_path):
    print("Reloading existing adapter from", adapter_dir)
    model = PeftModel.from_pretrained(model, adapter_dir)
    model.eval()
    train_stats = json.load(open(stats_path))
else:
    model, train_stats = L.train_lora(model, tok, train_all, LORA_CFG, out_dir=OUT_DIR)
print({k: v for k, v in train_stats.items() if k != "loss_log"})

In [ ]:
results["C4_lora"] = L.run_condition("C4_lora", model, tok, test_items, L.build_direct, L.score_text,
                                     OUT_DIR, batch_size=GEN_BATCH, max_new_tokens=MAX_NEW_TOK)

## 5 · Analysis

In [ ]:
import pandas as pd
LABELS = {"C1_direct": "C1 Zero-shot direct", "C2_cot": "C2 Zero-shot CoT", "C3_pal": "C3 PAL",
          "C4_lora": "C4 LoRA FT + direct", "C5_fewshot": "C5 4-shot"}
results = {k: results[k] for k in sorted(results)}   # C1, C2, C3, C4, (C5)
summary = L.summarize(results)
tests   = L.pairwise_tests(results)
df  = pd.DataFrame(summary).assign(label=lambda d: d.condition.map(LABELS))
dft = pd.DataFrame(tests)
display(df.round(4)); display(dft.round(4))

pal_status = pd.Series([r["status"] for r in results["C3_pal"]]).value_counts()
print("PAL execution status:\n", pal_status)

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 9})

fig, ax = plt.subplots(figsize=(3.5, 2.4))
acc = 100 * df.accuracy
err = [100 * (df.accuracy - df.ci_low), 100 * (df.ci_high - df.accuracy)]
ax.bar(df.label.str.split(" ").str[0], acc, yerr=err, capsize=3, color="#4C72B0")
ax.set_ylabel("Accuracy (%)"); ax.set_ylim(0, 100); ax.set_title("GSM8K accuracy (95% CI)")
fig.tight_layout(); fig.savefig(f"{OUT_DIR}/fig_accuracy.pdf"); fig.savefig(f"{OUT_DIR}/fig_accuracy.png", dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(3.5, 2.4))
ax.scatter(df.mean_gen_tokens, acc, color="#C44E52")
for _, r in df.iterrows():
    ax.annotate(r.label.split(" ")[0], (r.mean_gen_tokens, 100 * r.accuracy), textcoords="offset points", xytext=(4, 2))
ax.set_xlabel("Mean generated tokens"); ax.set_ylabel("Accuracy (%)"); ax.set_title("Accuracy vs. inference cost")
fig.tight_layout(); fig.savefig(f"{OUT_DIR}/fig_cost.pdf"); fig.savefig(f"{OUT_DIR}/fig_cost.png", dpi=200); plt.show()

if train_stats.get("loss_log"):
    lg = pd.DataFrame(train_stats["loss_log"])
    fig, ax = plt.subplots(figsize=(3.5, 2.2))
    ax.plot(lg.step, lg.loss, alpha=.35, color="#55A868")
    ax.plot(lg.step, lg.loss.rolling(10, min_periods=1).mean(), color="#55A868")
    ax.set_xlabel("Optimizer step"); ax.set_ylabel("Train loss"); ax.set_title("LoRA training loss")
    fig.tight_layout(); fig.savefig(f"{OUT_DIR}/fig_loss.pdf"); fig.savefig(f"{OUT_DIR}/fig_loss.png", dpi=200); plt.show()

In [ ]:
# Qualitative examples for the error-analysis section of the report.
by_idx = {k: {r["idx"]: r for r in v} for k, v in results.items()}
def pick(cond_right, cond_wrong, k=3):
    return [(by_idx[cond_right][i], by_idx[cond_wrong][i]) for i in by_idx[cond_right]
            if by_idx[cond_right][i]["correct"] and not by_idx[cond_wrong][i]["correct"]][:k]
examples = {
    "C4_right_C1_wrong": pick("C4_lora", "C1_direct"),
    "C1_right_C4_wrong": pick("C1_direct", "C4_lora"),
    "C3_right_C2_wrong": pick("C3_pal", "C2_cot"),
    "C2_right_C3_wrong": pick("C2_cot", "C3_pal"),
}
for k, v in examples.items():
    print("=" * 20, k, len(v))
    for a, b in v[:1]:
        print("Q:", a["question"][:300], "\nGOLD:", a["gold"])
        print("--- right:", a["output"][:500]); print("--- wrong:", b["output"][:500])

In [ ]:
final = {"env": ENV, "summary": summary, "pairwise_mcnemar": tests,
         "train": {k: v for k, v in train_stats.items() if k != "loss_log"},
         "pal_status": pal_status.to_dict(), "examples": examples}
json.dump(final, open(f"{OUT_DIR}/results.json", "w"), indent=2)
open(f"{OUT_DIR}/results_table.tex", "w").write(L.latex_results_table(summary, LABELS))
df.to_csv(f"{OUT_DIR}/summary.csv", index=False); dft.to_csv(f"{OUT_DIR}/mcnemar.csv", index=False)

# Small zip for the report: everything except the LoRA weights (those stay on Drive / can go to the HF Hub).
import zipfile
zip_path = "/content/YZM5109_A1_results_small.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, fnames in os.walk(OUT_DIR):
        if "lora_adapter" in root:
            continue
        for fn in fnames:
            full = os.path.join(root, fn)
            z.write(full, os.path.relpath(full, OUT_DIR))
print("Saved:", zip_path, f"{os.path.getsize(zip_path) / 1e6:.1f} MB")
try:
    from google.colab import files; files.download(zip_path)
except Exception as e:
    print("Download manually from the Files panel:", e)

## 5b · Compact results to copy-paste (if you cannot upload files)

In [ ]:
compact = {"env": ENV, "summary": summary, "pairwise_mcnemar": tests,
           "train": {k: v for k, v in train_stats.items() if k != "loss_log"},
           "pal_status": pal_status.to_dict(),
           "loss_every_10": [round(x["loss"], 4) for x in train_stats.get("loss_log", [])][::10],
           "examples": {k: [{"q": a["question"][:400], "gold": a["gold"], "right": a["output"][:600],
                             "wrong": b["output"][:600]} for a, b in v[:2]] for k, v in examples.items()}}
print(json.dumps(compact, default=float))

## 6 · (Optional) Push the LoRA adapter to the Hugging Face Hub
Add an `HF_TOKEN` secret in Colab (key icon on the left) with *write* access, then run.

In [ ]:
PUSH_TO_HUB = False
HUB_REPO    = "your-username/qwen2.5-1.5b-gsm8k-lora-r16"
if PUSH_TO_HUB:
    from google.colab import userdata
    from huggingface_hub import login
    login(userdata.get("HF_TOKEN"))
    model.push_to_hub(HUB_REPO, private=True)
    tok.push_to_hub(HUB_REPO, private=True)